# Task 1: Target Selection & Scraping

> 1.1 Data Selection: Choose one disaster category (Earthquakes, Tsunamis, Floods, Cyclones, Volcanoes, Droughts, or Forest Fires) and choose two different regions.




> 1.2 The "Source of Truth" Scraper: Identify two specific events from those two regions:
> * Event A: A "Historical" event (at least 5–8 years old).
> * Event B: A "Recent" event (within the last 12 months).

In [ ]:
import requests

URL = "https://www.gdacs.org/Alerts/default.aspx"
page = requests.get(URL)

print(page.text)



<!DOCTYPE html>

<html xmlns="http://www.w3.org/1999/xhtml">
<head><title>
	GDACS - Global Disaster Awareness and Coordination System
</title>
    <link rel="stylesheet" href="../scripts/openlayers/3.20.1/ol.css" type="text/css" />
    <link rel="stylesheet" href="../scripts/openlayers/layerswitcher/ol3-layerswitcher.css" />
    <link href="../Styles/gdacs_mapstyle.css" rel="stylesheet" />
    <link rel="stylesheet" href="../Content/all.css" />
    <script type="text/javascript" src="../scripts/openlayers/3.20.1/ol.js"></script>
    <script type="text/javascript" src="../scripts/openlayers/layerswitcher/ol3-layerswitcher.js"></script>
    <script type="text/javascript" src="../Scripts/Search/searchtools.js"></script>
    <link href="../Styles/gdacs_2.0_bootstrap4to5.css" rel="stylesheet" />

<link href="/Content/bootstrap.min.css" rel="stylesheet" /><link href="/Styles/gdacs_2.0.css" rel="stylesheet" /><meta name="viewport" content="width=device-width, initial-scale=1.0" /><link rel=

## Source of Truth Scraper

Learning notes from building this scraper:
* Instead of treating the dropdown as a set of clickable buttons, one needs to use the Select class. 
* Dropdown was especially difficult to work with as it is a JavaScript (dynamic) element that can't directly be interacted with, even after wait times. Tried using a select by value, visible text and index, and none of them directly worked.
* Incorporated wait times in case it was a DOM loading issue
* Finally, realised that it needed a direct JS script since this was a dynamic element.

Refs: https://realpython.com/modern-web-automation-with-python-and-selenium/,  https://www.selenium.dev, https://www.selenium.dev/selenium/docs/api/java/org/openqa/selenium/safari/SafariDriver.html

Would want to make this scraper **more modular and generalisable** so that I can call it on different countries, to and from dates (strings) and checkboxes.

In [ ]:
jobs = [
    ("inputChFl", "Bangladesh", "2025-02-25", "2026-02-25"), # recent (2025-2026)
    ("inputChFl", "Bangladesh", "2018-02-25", "2024-02-25"), # historical (2018-2024)
    ("inputChFl", "Italy", "2025-02-25", "2026-02-25"), # recent 
    ("inputChFl", "Italy", "2018-02-25", "2024-02-25"), # historical
]

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)
df

In [ ]:
from pathlib import Path
from datetime import datetime

save_dir = Path("data")
save_dir.mkdir(exist_ok=True)

latest_csv_path = save_dir / "gdacs_df_latest.csv"
versioned_csv_path = save_dir / f"gdacs_df_{datetime.now():%Y%m%d_%H%M%S}.csv"

df.to_csv(latest_csv_path, index=False)
df.to_csv(versioned_csv_path, index=False)

print("Saved latest:", latest_csv_path)
print("Saved versioned:", versioned_csv_path)

# INBUILT VERSIONING APPROACH: save with timestamped filename, and also overwrite a "latest" file for easy access.
# Note: to reload later in any new session, have to do:
# import pandas as pd
# df = pd.read_csv("data/gdacs_df_latest.csv")

### Additional insights I'm extracting from each tab:
- Event summary table fields: GDACS ID, deaths, displaced, and event date range (From-To).
- GDACS score panel: numeric score value shown in the gauge and model info link.
- Impact context signals: presence of exposed-population section and available map/layer context.
- Activation/operations indicators: Virtual OSOCC, Meteo assessment, Satellite products, Analytical products.
- Media context: casualties mention count and last-hour article count from the media summary line.
- Description/meta: headline title, bulletin date, provider/source link.
- External data-source inventory from resources block (e.g., EC-JRC, NASA, WMO, INFORM counts).

Having successfully run the scraper on the site a few times, now choosing the "data/gdacs_df_20260225_190435.csv" version of the output (CSV file) to run the advanced deep tab scraper.

### Summary Tab

In [ ]:
deep_data_list = []
for idx, row in df.iterrows():
    link = row.get("random_link")
    if isinstance(link, str) and link.strip():
        deep_data = scrape_gdacs_deep_tabs(link.strip(), headless=True)
        deep_data_list.append(deep_data)
    else:
        deep_data_list.append({"link": link})
deep_data_list

[{'link': 'https://www.gdacs.org/report.aspx?eventid=1103509&episodeid=1&eventtype=FL',
  'gdacs_id': 'FL 1103509',
  'death_count': 2,
  'displaced_count': None,
  'countries': 'Bangladesh',
  'event_from_to': '18 Sep - 20 Sep',
  'articles_count': 10,
  'about_casualties_count': 0,
  'last_hour_count': 0,
  'news_items': '[{"title": "Ganga Basin Faced Severe Drying Over The Last 30 Years Due To Frequent..", "link": "https://www.news18.com/india/ganga-basin-faced-severe-drying-over-the-last-30-years-due-to-frequent-droughts-shows-new-study-ws-l-9591630.html", "date": "9/23/2025", "time": "11:06:00AM", "first_paragraph": "The Ganga River Basin, lifeline of over 600 million people, experienced its most severe drying trend from 1991 to 2020 \\u2013 the worst in the past 1,000 years, according to a new research, which examined 1,300 years of streamflow data and the i\\t\\t\\t ...(more)\\t\\t"}, {"title": "Saudi Arabia and Pakistan just formed a defence pact\\u2014India must rethin..", "li

The following cell was used for versioning to test the scrapers, but need not be used now:

In [ ]:
deep_df = pd.DataFrame(deep_data_list)
deep_df.to_csv("deep_tab_outputs/summary.csv", index=False)

In [ ]:
# Compactly viewing the deep_df
priority_cols = [
    "country",
    "report_country",
    "date_from",
    "date_to",
    "random_link",
    "magnitude",
    "alert_level",
    "gdacs_score",
    "gdacs_id",
    "death_count",
    "displaced_count",
    "population_exposed_100km",
    "vulnerability_inform_index",
    "secondary_risks",
    "news_articles_count",
    "about_casualties_count",
    "last_hour_count",
    "social_media_mentions_count",
    "activation_tabs",
    "description_title"
    ]

report_cols = [c for c in priority_cols if c in df.columns]
report_df = df[report_cols].copy()

sort_cols = [c for c in ["country", "report_country", "date_from"] if c in report_df.columns]
if sort_cols:
    report_df = report_df.sort_values(sort_cols).reset_index(drop=True)

print("Reporting columns:", report_cols)
report_df

Reporting columns: ['country', 'date_from', 'date_to', 'random_link']


Reporting columns: ['country', 'date_from', 'date_to', 'random_link']


,country,date_from,date_to,random_link
0,Bangladesh,2018-02-25,2024-02-25,https://www.gdacs.org/report.aspx?eventid=1100...
1,Bangladesh,2025-02-25,2026-02-25,https://www.gdacs.org/report.aspx?eventid=1103...
2,Italy,2018-02-25,2024-02-25,https://www.gdacs.org/report.aspx?eventid=1101...
3,Italy,2025-02-25,2026-02-25,https://www.gdacs.org/report.aspx?eventid=1103...


### Media Tab

In [ ]:
# Define a function to scrape media tab for a single event
def scrape_media_for_event(gdacs_id, driver, wait):
    if not gdacs_id:
        print("Invalid gdacs_id provided.")
        return {"gdacs_id": gdacs_id, "media_articles": []}
    try:
        # Robustly extract eventid and eventtype
        if isinstance(gdacs_id, str) and " " in gdacs_id:
            eventtype, eventid = gdacs_id.split()
        else:
            eventtype = "FL"
            eventid = str(gdacs_id)
        # Compose the media tab URL for debugging
        media_url = f"https://www.gdacs.org/media.aspx?eventid={eventid}&episodeid=1&eventtype={eventtype}"
        print(f"DEBUG: Would scrape media tab at: {media_url}")
        driver.get(media_url)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        articles = []
        # Find all media_news_box divs
        news_boxes = driver.find_elements(By.CSS_SELECTOR, "#emmList .media_news_box")
        print(f"Found {len(news_boxes)} media_news_box elements")
        for box in news_boxes:
            try:
                title_el = box.find_element(By.CSS_SELECTOR, ".p_media_title a")
                title = title_el.text.strip()
                link = title_el.get_attribute("href")
            except Exception as e:
                print(f"Error extracting title/link: {e}")
                title = ""
                link = ""
            try:
                date_source = box.find_element(By.CSS_SELECTOR, ".p_media_dida").text.strip()
            except Exception as e:
                print(f"Error extracting date_source: {e}")
                date_source = ""
            try:
                summary = box.find_element(By.CSS_SELECTOR, ".p_summary").text.strip()
            except Exception as e:
                print(f"Error extracting summary: {e}")
                summary = ""
            articles.append({
                "title": title,
                "link": link,
                "date_source": date_source,
                "summary": summary
            })
        return {"gdacs_id": gdacs_id, "media_articles": articles}
    except Exception as e:
        print(f"Error processing gdacs_id {gdacs_id}: {e}")
        return {"gdacs_id": gdacs_id, "media_articles": []}

# Main script to process all rows in summary.csv
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pathlib import Path

# Load summary.csv
input_file = "deep_tab_outputs/summary.csv"
try:
    summary_df = pd.read_csv(input_file)
    if "gdacs_id" not in summary_df.columns:
        raise ValueError("Input file does not contain the required 'gdacs_id' column.")
    print("Loaded summary DataFrame:")
    print(summary_df.head())
except FileNotFoundError:
    print(f"Error: Input file '{input_file}' not found.")
    raise
except Exception as e:
    print(f"Error loading input file: {e}")
    raise

media_results = []

# Initialize WebDriver once for all events
options = webdriver.SafariOptions()
options.add_argument("--headless")
driver = webdriver.Safari(options=options)
wait = WebDriverWait(driver, 20)  # Increased timeout to 20 seconds

try:
    for idx, row in summary_df.iterrows():
        gdacs_id = row.get("gdacs_id")
        print(f"Processing row {idx+1}/{len(summary_df)}: gdacs_id: {gdacs_id}")
        try:
            result = scrape_media_for_event(gdacs_id, driver, wait)
        except Exception as e:
            print(f"Exception for row {idx+1}: {e}")
            result = {"gdacs_id": gdacs_id, "media_articles": []}
        media_results.append(result)
finally:
    driver.quit()

# Convert to DataFrame for analysis
media_flat = []
for event in media_results:
    event_id = event["gdacs_id"]
    for article in event["media_articles"]:
        media_flat.append({"gdacs_id": event_id, **article})
media_df = pd.DataFrame(media_flat)

print("Scraped media_df shape:", media_df.shape)
print(media_df.head())

# Save to CSV for further analysis
output_dir = Path("deep_tab_outputs")
output_dir.mkdir(exist_ok=True)
media_csv = output_dir / "media.csv"
media_df.to_csv(media_csv, index=False)
print("Saved media tab to:", media_csv)

Loaded summary DataFrame:
                                                link    gdacs_id  death_count  \
0  https://www.gdacs.org/report.aspx?eventid=1103...  FL 1103509            2   
1  https://www.gdacs.org/report.aspx?eventid=1100...  FL 1100541           67   
2  https://www.gdacs.org/report.aspx?eventid=1103...  FL 1103166            0   
3  https://www.gdacs.org/report.aspx?eventid=1101...  FL 1101944            0   

   displaced_count   countries    event_from_to  articles_count  \
0              NaN  Bangladesh  18 Sep - 20 Sep              10   
1          56000.0  Bangladesh  28 Jun - 10 Jul              68   
2           1084.0       Italy  14 Mar - 30 Mar              40   
3             48.0       Italy  15 Apr - 17 Apr               9   

   about_casualties_count  last_hour_count  \
0                       0                0   
1                      16                0   
2                       0                0   
3                       0                0   

 

In [ ]:
# Diagnostics: Check if all events were processed and file was written
import pandas as pd
from pathlib import Path

# Check media_results and media_df
print(f"Number of events processed (media_results): {len(media_results)}")
print(f"Number of rows in media_df: {len(media_df)}")
print("First few rows of media_df:")
print(media_df.head())

# Check if media.csv exists and print its contents
media_csv = Path("deep_tab_outputs/media.csv")
if media_csv.exists():
    print("media.csv exists. First few rows:")
    print(pd.read_csv(media_csv).head())
else:
    print("media.csv does not exist!")


Number of events processed (media_results): 4
Number of rows in media_df: 10
First few rows of media_df:
     gdacs_id                                              title  \
0  FL 1103509  Breakthrough in cold-tolerant rice development...   
1  FL 1103509  Ganga Basin Faced Severe Drying Over The Last ...   
2  FL 1103509  Saudi Arabia and Pakistan just formed a defenc...   
3  FL 1103509       Signals lowered at seaports, fresh low brews   
4  FL 1103509             CM meets envoys of Turkiye, Bangladesh   

                                                link  \
0  https://www.cgiar.org/news-events/news/breakth...   
1  https://www.news18.com/india/ganga-basin-faced...   
2  https://www.livemint.com/opinion/online-views/...   
3       https://bdnews24.com/bangladesh/b6fb169e7a6a   
4  https://www.brecorder.com/news/40384037/cm-mee...   

                      date_source  \
0      2025-09-23T20:27:00(cgiar)   
1     2025-09-23T11:06:00(news18)   
2   2025-09-23T09:30:00(livemint)   
3

In [ ]:
# do an inner join on data/gdacs_df_20260225_215447.csv, summary.csv, media.csv, impact.csv so all of the info is in one csv file, then save as data/outputs/20260225_215447.csv

# Task 1: ReliefWeb

In [87]:
import requests

URL = "https://reliefweb.int/disasters"
page = requests.get(URL)

print(page.text)

<!DOCTYPE html>
<html lang="en" dir="ltr" prefix="og: https://ogp.me/ns#" class="no-js">
  <head>
    <meta charset="utf-8" />
<script>document.documentElement.className = document.documentElement.className.replace('no-js', typeof document.addEventListener !== 'undefined' ? 'js' : 'no-js');</script>
<meta name="description" content="ReliefWeb disaster pages provide an overview of the situation and situation reports, news and press releases, assessments, evaluations, infographics and maps. Browse our list of natural disasters with humanitarian impact from 1981 until today." />
<link rel="canonical" href="https://reliefweb.int/disasters" />
<link rel="image_src" href="https://reliefweb.int/modules/custom/reliefweb_meta/images/default.png" />
<meta property="og:site_name" content="ReliefWeb" />
<meta property="og:type" content="article" />
<meta property="og:url" content="https://reliefweb.int/disasters" />
<meta property="og:title" content="Disasters" />
<meta property="og:description" c

In [46]:
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_reliefweb(country, from_date, to_date):
    URL = "https://reliefweb.int/disasters"
    driver = webdriver.Safari()
    driver.get(URL)
    wait = WebDriverWait(driver, 10)

    # COUNTRY
    # Find and click the Country button
    country_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Add Country' and @data-field='C' and @data-operator='or']/span[text()='Country']/..")))
    country_button.click()

    # Send country string keys to input field
    country_input = wait.until(EC.presence_of_element_located((By.ID, "river-advanced-search-autocomplete-widget-C")))
    country_input.send_keys(country)

    # Wait for the dropdown
    time.sleep(3)

    # Press down and enter buttons to select country
    country_input.send_keys("\ue015")  # Arrow down key
    country_input.send_keys("\ue007")  # Enter key

    # DISASTER TYPE

    # Click the Disaster Type button
    disaster_type_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Add Disaster type' and @data-field='TY' and @data-operator='or']/span[text()='Disaster type']/..")))
    disaster_type_button.click()

    # Select "Flood" from the dropdown
    dropdown = wait.until(EC.presence_of_element_located((By.ID, "river-advanced-search-options-widget-TY")))
    select = Select(dropdown)
    select.select_by_value("4611")  # 4611 is the value for Flood

    # Add
    add_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@data-add='']")))
    add_button.click()

    # STATUS
    # Click the Status button
    status_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Add Status' and @data-field='ST' and @data-operator='or']/span[text()='Status']/..")))
    status_button.click()

    # Select "Past disaster" from the dropdown
    dropdown_status = wait.until(EC.presence_of_element_located((By.ID, "river-advanced-search-options-widget-ST")))
    select_status = Select(dropdown_status)
    select_status.select_by_value("past")  # 'past' is the value for Past disaster
    
    # Click the "Add" button
    add_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@data-add='']")))
    add_button.click()

    # DATE
    # Click the Date button
    date_button = wait.until(EC.element_to_be_clickable((
        By.XPATH,
        "//button[@aria-label='Add Date' and @data-field='DA']"
    )))
    date_button.click()

    # Wait for date widget to appear and find the from and to fields
    from_field = wait.until(EC.presence_of_element_located((
        By.ID,
        "river-advanced-search-date-widget-DA-from"
    )))

    to_field = wait.until(EC.presence_of_element_located((
        By.ID,
        "river-advanced-search-date-widget-DA-to"
    )))

    # Clear and type
    from_field.clear()
    from_field.send_keys(from_date)

    to_field.clear()
    to_field.send_keys(to_date)

    # Add
    add_button = wait.until(EC.element_to_be_clickable((
        By.XPATH,
        "//button[@data-add='']"
    )))
    add_button.click()

    # Store old first article BEFORE clicking Apply
    old_first_article = driver.find_element(
        By.CSS_SELECTOR,
        "article.rw-river-article"
    )

    # Apply filters
    apply_button = wait.until(EC.element_to_be_clickable((
        By.XPATH,
        "//button[@data-apply='true']"
    )))
    apply_button.click()

    # Wait until old results disappear (page refresh via AJAX)
    wait.until(EC.staleness_of(old_first_article))

    # Now wait for new filtered results to load
    wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR,
        "div.rw-river__articles article.rw-river-article"
    )))

    # ARTICLES

    # Wait until at least one article loads 
    first_article = wait.until(
        EC.presence_of_element_located((
            By.CSS_SELECTOR,
            "div.rw-river__articles article.rw-river-article"
        ))
    )

    #   Basic attributes from <article> tag  
    data_id = first_article.get_attribute("data-id")
    disaster_type_code = first_article.get_attribute("data-disaster-type")
    disaster_status_code = first_article.get_attribute("data-disaster-status")

    #   Title and URL  
    title_element = first_article.find_element(
        By.CSS_SELECTOR,
        "h3.rw-river-article__title a"
    )

    title = title_element.text.strip()
    url = title_element.get_attribute("href")

    #   Status text  
    status = first_article.find_element(
        By.CSS_SELECTOR,
        "dd.rw-entity-meta__tag-value--status"
    ).text.strip()

    #   Disaster types (may be multiple)  
    disaster_types_elements = first_article.find_elements(
        By.CSS_SELECTOR,
        "dd.rw-entity-meta__tag-value--disaster-type li a"
    )

    disaster_types = [e.text.strip() for e in disaster_types_elements]

    #   Country  
    country = first_article.find_element(
        By.CSS_SELECTOR,
        "dd.rw-entity-meta__tag-value--country li a"
    ).text.strip()

    #   Build DataFrame row  
    relief_df = pd.DataFrame([{
        "data_id": data_id,
        "disaster_type_code": disaster_type_code,
        "disaster_status_code": disaster_status_code,
        "title": title,
        "url": url,
        "status_text": status,
        "disaster_types": disaster_types,
        "country": country
    }])

    print(relief_df)

    driver.quit()

    return relief_df

In [47]:
relief_results = pd.DataFrame()

relief_results = pd.concat([
    relief_results,
    scrape_reliefweb("Bangladesh", "2025-02-25", "2026-02-25"),
    scrape_reliefweb("Bangladesh", "2018-02-25", "2024-02-25"),
    scrape_reliefweb("Yemen", "2025-02-25", "2026-02-25"),
    scrape_reliefweb("Yemen", "2018-02-25", "2024-02-25")
], ignore_index=True)

print(relief_results)

  data_id disaster_type_code disaster_status_code  \
0   52376                 FL                 past   

                                          title  \
0  Bangladesh: Floods and Landslides - May 2025   

                                                 url    status_text  \
0  https://reliefweb.int/disaster/fl-2025-000076-bgd  Past Disaster   

  disaster_types     country  
0        [Flood]  Bangladesh  
  data_id disaster_type_code disaster_status_code  \
0   51682                 FL                 past   

                                          title  \
0  Bangladesh: Floods and Landslides - Aug 2023   

                                                 url    status_text  \
0  https://reliefweb.int/disaster/fl-2023-000157-bgd  Past Disaster   

  disaster_types     country  
0  [Flash Flood]  Bangladesh  
  data_id disaster_type_code disaster_status_code                     title  \
0   52427                 FL                 past  Yemen: Floods - Aug 2025   

           

In [48]:
relief_results.to_csv("data/used/reliefweb_results.csv", index=False)

In [49]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def enrich_disaster_pages(relief_df):

    driver = webdriver.Safari()
    wait = WebDriverWait(driver, 10)

    enriched_rows = []

    for _, row in relief_df.iterrows():

        disaster_url = row["url"]
        print(f"Processing: {disaster_url}")

        try:
            driver.get(disaster_url)

            # ---------------------------
            # DISASTER DESCRIPTION
            # ---------------------------
            wait.until(EC.presence_of_element_located((By.ID, "overview-content")))
            disaster_description = driver.find_element(
                By.ID, "overview-content"
            ).text.strip()

            # ---------------------------
            # AFFECTED COUNTRIES
            # ---------------------------
            affected_countries = []

            try:
                wait.until(EC.presence_of_element_located((By.ID, "countries")))

                country_elements = driver.find_elements(
                    By.CSS_SELECTOR,
                    "section#countries article.rw-river-article--country h3 a"
                )

                affected_countries = list(set(
                    [c.text.strip() for c in country_elements]
                ))

            except:
                affected_countries = []

            # ---------------------------
            # UPDATES LINK
            # ---------------------------
            updates_link = None

            try:
                updates_link = driver.find_element(
                    By.CSS_SELECTOR,
                    "footer.rw-river__view-more a"
                ).get_attribute("href")
            except:
                updates_link = None

            # ---------------------------
            # BUILD ROW
            # ---------------------------
            enriched_row = row.to_dict()
            enriched_row["disaster_description"] = disaster_description
            enriched_row["affected_countries"] = affected_countries
            enriched_row["updates_link"] = updates_link

            enriched_rows.append(enriched_row)

        except Exception as e:
            print(f"Failed on {disaster_url}: {e}")

    driver.quit()

    return pd.DataFrame(enriched_rows)

In [50]:
reliefweb_enriched_df = enrich_disaster_pages(relief_results)

print(reliefweb_enriched_df.head())

Processing: https://reliefweb.int/disaster/fl-2025-000076-bgd
Processing: https://reliefweb.int/disaster/fl-2023-000157-bgd
Processing: https://reliefweb.int/disaster/fl-2025-000151-yem
Processing: https://reliefweb.int/disaster/tc-2023-000212-yem
  data_id disaster_type_code disaster_status_code  \
0   52376                 FL                 past   
1   51682                 FL                 past   
2   52427                 FL                 past   
3   51780                 TC                 past   

                                          title  \
0  Bangladesh: Floods and Landslides - May 2025   
1  Bangladesh: Floods and Landslides - Aug 2023   
2                      Yemen: Floods - Aug 2025   
3               Tropical Cyclone Tej - Oct 2023   

                                                 url    status_text  \
0  https://reliefweb.int/disaster/fl-2025-000076-bgd  Past Disaster   
1  https://reliefweb.int/disaster/fl-2023-000157-bgd  Past Disaster   
2  https://relief

In [51]:
reliefweb_enriched_df.to_csv("data/used/reliefweb_enriched_results.csv", index=False)

This^ csv saves the "deeptab" outputs. When you click an event page, you get its description, affected country and a link to media updates. This enriched_results csv saves all this metadata alongside the event. Now for another scrape of the media link!